# VRDU Registration Form CBA Experiment

**Domain**: Government Registration Forms (FARA)  
**Method**: Vision extraction — LLM reads FARA registration form images and maps values to canonical concepts  
**Models**: Claude Haiku 4.5, GPT-4o-mini  
**Scoring**: Value-first CBA matching

Foreign Agents Registration Act (FARA) forms require agents representing foreign entities to file
with the US Department of Justice. Each form contains three confusable name fields:
the registrant (US-based agent), the foreign principal (entity being represented),
and the signer (person who signed the form). The model must correctly bind each name to its role.

Key confusable pairs: registrant_name ↔ foreign_principle_name ↔ signer_name (all person/org names),
file_date vs other dates in the document body.

Source: [VRDU (Google Research)](https://github.com/google-research-datasets/vrdu) — 1,915 real FARA registration forms.

In [ ]:
import subprocess, sys
for pkg in ["anthropic", "openai", "pymupdf", "Pillow"]:
    mod = {"Pillow": "PIL", "pymupdf": "fitz"}.get(pkg, pkg)
    try:
        __import__(mod)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "--break-system-packages", "-q"])

import anthropic, openai, json, time, os, random, io, base64, gzip
import fitz  # PyMuPDF
from PIL import Image as PILImage
from collections import defaultdict, Counter
from google.colab import userdata


ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

os.environ['ANTHROPIC_API_KEY'] = ANTHROPIC_API_KEY
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

print("Setup complete")

Setup complete


In [ ]:
import subprocess, sys, os
import shutil # Added for shutil.copyfileobj

# --- Download VRDU registration forms ---
VRDU_BASE = "/tmp/vrdu"
VRDU_PATH = os.path.join(VRDU_BASE, "registration-form", "main") # This is the FARA forms path
PDF_DIR = os.path.join(VRDU_PATH, "pdfs")

if not os.path.isdir(PDF_DIR) or len(os.listdir(PDF_DIR)) < 1900:
    print("Downloading VRDU dataset (registration forms)...")
    print("This is ~400 MB and may take a few minutes on first run.")
    if os.path.exists(VRDU_BASE):
        shutil.rmtree(VRDU_BASE)
    # Shallow clone — downloads only latest commit, all files
    subprocess.run([
        "git", "clone", "--depth=1",
        "https://github.com/google-research-datasets/vrdu.git", VRDU_BASE
    ], check=True)
    print("Clone complete.")
else:
    print("VRDU registration data already present.")

# Verify
n_pdfs = len([f for f in os.listdir(PDF_DIR) if f.endswith(".pdf")])
assert n_pdfs >= 1900, f"Expected ~1915 PDFs, found {n_pdfs}. Clone may have failed."

# Decompress dataset.jsonl.gz if needed
dataset_path = os.path.join(VRDU_PATH, "dataset.jsonl")
if not os.path.exists(dataset_path):
    gz_path = dataset_path + ".gz"
    with gzip.open(gz_path, 'rb') as f_in:
        with open(dataset_path, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
    print(f"Decompressed dataset.jsonl")

print(f"VRDU path: {VRDU_PATH}")
print(f"PDFs: {n_pdfs} files")

# 9 unrepeated (document-level) concepts to evaluate
EVAL_CONCEPTS = [
    "registration_num",       # FARA registration number
    "registrant_name",        # US-based agent/firm name
    "foreign_principle_name", # Foreign entity being represented
    "signer_name",            # Person who signed the form
    "signer_title",           # Title of the signer
    "file_date",              # Date of filing/signing
]

# --- Diagnostic: check PDF directory ---
print(f"PDF_DIR: {PDF_DIR}")
print(f"PDF_DIR exists: {os.path.isdir(PDF_DIR)}")
if os.path.isdir(PDF_DIR):
    pdf_files_on_disk = [f for f in os.listdir(PDF_DIR) if f.endswith(".pdf")]
    print(f"PDFs on disk: {len(pdf_files_on_disk)}")
else:
    pdf_files_on_disk = []
    print("WARNING: PDF directory not found!")

# Parse dataset
all_docs = []
total_parsed = 0
enough_concepts = 0
pdf_found = 0

with open(dataset_path) as f:
    for line in f:
        doc = json.loads(line)
        total_parsed += 1
        gt = {}
        for ann in doc["annotations"]:
            entity_name = ann[0]
            if isinstance(entity_name, str) and entity_name in EVAL_CONCEPTS:
                values = ann[1]
                if values:
                    text = values[0][0].strip()
                    if text and entity_name not in gt:
                        gt[entity_name] = text

        n_pages = len(doc["ocr"]["pages"])
        pdf_path = os.path.join(PDF_DIR, doc["filename"])

        has_concepts = len(gt) >= 3 # Using >= 3 concepts for FARA forms
        has_pdf = os.path.exists(pdf_path)

        if has_concepts:
            enough_concepts += 1
        if has_pdf:
            pdf_found += 1

        if has_concepts and has_pdf:
            all_docs.append({
                "filename": doc["filename"],
                "pdf_path": pdf_path,
                "ground_truth": gt,
                "n_pages": n_pages,
            })

print(f"\n--- Loading Summary ---")
print(f"Total docs in jsonl: {total_parsed}")
print(f"Docs with >= 3 concepts: {enough_concepts}")
print(f"Docs with PDF on disk: {pdf_found}")
print(f"Docs passing both filters: {len(all_docs)}")

if len(all_docs) == 0 and enough_concepts > 0 and pdf_found == 0:
    print(f"\nDIAGNOSTIC: Concepts OK but no PDFs found.")
    # The following line might raise error if dataset_path is empty
    # print(f"First jsonl filename: {json.loads(open(dataset_path).readline())['filename']}")
    if pdf_files_on_disk:
        print(f"First PDF on disk: {pdf_files_on_disk[0]}")
    else:
        print("No PDFs in PDF_DIR — git clone may not have downloaded them.")
        print("Try running: !git clone --depth=1 https://github.com/google-research-datasets/vrdu.git /tmp/vrdu")
elif len(all_docs) > 0:
    print(f"\nConcept coverage:")
    for c in EVAL_CONCEPTS:
        n = sum(1 for d in all_docs if c in d["ground_truth"])
        print(f"  {c:<20s}: {n}/{len(all_docs)} ({n/len(all_docs):.0%})")
    print(f"\nSample GT (first doc):")
    for k, v in all_docs[0]["ground_truth"].items():
        print(f"  {k}: {v}")
else:
    print("No documents passed filters. Check dataset integrity and paths.")

This is ~400 MB and may take a few minutes on first run.
Clone complete.
Decompressed dataset.jsonl
VRDU path: /tmp/vrdu/registration-form/main
PDFs: 1915 files
PDF_DIR: /tmp/vrdu/registration-form/main/pdfs
PDF_DIR exists: True
PDFs on disk: 1915

--- Loading Summary ---
Total docs in jsonl: 1915
Docs with >= 3 concepts: 1902
Docs with PDF on disk: 1915
Docs passing both filters: 1902

Concept coverage:
  registration_num    : 1892/1902 (99%)
  registrant_name     : 1890/1902 (99%)
  foreign_principle_name: 1132/1902 (60%)
  signer_name         : 1467/1902 (77%)
  signer_title        : 549/1902 (29%)
  file_date           : 1871/1902 (98%)

Sample GT (first doc):
  registration_num: 3712
  registrant_name: DLA Piper US LLP
  file_date: July 16, 2008


In [ ]:
FARA_ONTOLOGY = {
    "families": {
        "names": ["registrant_name", "foreign_principle_name", "signer_name"],
        "signer_details": ["signer_title"],
        "identifiers": ["registration_num"],
        "dates": ["file_date"],
    }
}

CONCEPT_TO_FAMILY = {c: f for f, cs in FARA_ONTOLOGY["families"].items() for c in cs}

FARA_SYSTEM_PROMPT = """You are a document AI extraction system. You extract structured data from FARA (Foreign Agents Registration Act) registration form images.

Given an image (or multiple pages) of a FARA registration/amendment form, extract values for the following canonical concept keys.

CANONICAL CONCEPT KEYS:
- registration_num: The FARA registration number (numeric, usually near the top of the form)
- registrant_name: Name of the registering agent — the US-based law firm, lobbying organization, or individual doing the representation
- foreign_principle_name: Name of the foreign government, organization, company, or entity being represented
- signer_name: Name of the person who signed the form (found in the execution/signature section)
- signer_title: Title or position of the person who signed (e.g., "Partner", "Managing Director")
- file_date: Date the form was filed or signed (found near the signature)

IMPORTANT DISAMBIGUATION RULES:
- registrant_name is the AGENT (US-based entity doing lobbying/representation work)
- foreign_principle_name is the FOREIGN entity being represented (a government, foreign company, or foreign organization)
- signer_name is the specific PERSON who signed, who may be a partner/employee of the registrant
- file_date is the date of signing/filing, NOT other dates mentioned in the document body

RULES:
1. Return ONLY a JSON object with exactly these 6 keys.
2. If a field is not present or you cannot determine it, use "N/A".
3. Return ONLY valid JSON, no other text."""

print(f"Ontology: {len(EVAL_CONCEPTS)} concepts, {len(FARA_ONTOLOGY['families'])} families")
for fam, concepts in FARA_ONTOLOGY["families"].items():
    print(f"  {fam}: {concepts}")

Ontology: 6 concepts, 4 families
  names: ['registrant_name', 'foreign_principle_name', 'signer_name']
  signer_details: ['signer_title']
  identifiers: ['registration_num']
  dates: ['file_date']


In [ ]:
SAMPLE_SIZE = 50
random.seed(42)

indices = list(range(len(all_docs)))
random.shuffle(indices)
selected = sorted(indices[:SAMPLE_SIZE])

samples = []
for i, idx in enumerate(selected):
    doc = all_docs[idx]
    samples.append({
        "doc_id": f"fara_{i:04d}",
        "filename": doc["filename"],
        "pdf_path": doc["pdf_path"],
        "ground_truth": doc["ground_truth"],
        "n_pages": doc["n_pages"],
    })

filled = Counter()
for s in samples:
    for c in EVAL_CONCEPTS:
        if c in s["ground_truth"]:
            filled[c] += 1

print(f"Sampled {len(samples)} FARA registration forms")
print(f"\nField fill rates:")
for c in EVAL_CONCEPTS:
    print(f"  {c:<25s}: {filled[c]}/{SAMPLE_SIZE} ({filled[c]/SAMPLE_SIZE:.0%})")

page_dist = Counter(s["n_pages"] for s in samples)
print(f"\nPage count distribution:")
for k in sorted(page_dist.keys()):
    print(f"  {k} pages: {page_dist[k]} docs")

Sampled 50 FARA registration forms

Field fill rates:
  registration_num         : 49/50 (98%)
  registrant_name          : 50/50 (100%)
  foreign_principle_name   : 29/50 (58%)
  signer_name              : 38/50 (76%)
  signer_title             : 15/50 (30%)
  file_date                : 49/50 (98%)

Page count distribution:
  1 pages: 2 docs
  2 pages: 45 docs
  3 pages: 3 docs


In [ ]:
def normalize_value(v):
    """Normalize a value for comparison."""
    v = str(v).strip().lower()
    v = " ".join(v.split())
    return v

def score_document(gt, pred):
    """Score a single document with value-first CBA matching."""
    pred_norm = {k: normalize_value(v) for k, v in pred.items()}
    pred_values = {v for v in pred_norm.values() if v != normalize_value("N/A")}

    per_field = {}
    field_correct = 0
    cba_correct = 0
    scored = []

    for concept in EVAL_CONCEPTS:
        gt_val = gt.get(concept)
        if gt_val is None:
            continue

        gt_norm = normalize_value(gt_val)
        pred_val = normalize_value(pred.get(concept, "N/A"))

        scored.append(concept)

        # CBA: value under correct concept key
        cba_match = (gt_norm == pred_val)

        # Also check containment for names (GT may be substring or vice versa)
        if not cba_match and concept in ("registrant_name", "foreign_principle_name", "signer_name", "signer_title"):
            cba_match = (gt_norm in pred_val) or (pred_val in gt_norm and len(pred_val) > 3)

        # Field Recall: value found anywhere in predictions
        field_match = (gt_norm in pred_values)
        if not field_match and concept in ("registrant_name", "foreign_principle_name", "signer_name", "signer_title"):
            for pv in pred_values:
                if (gt_norm in pv) or (pv in gt_norm and len(pv) > 3):
                    field_match = True
                    break

        if cba_match:
            cba_correct += 1
        if field_match:
            field_correct += 1

        bound_to = None
        if field_match and not cba_match:
            for k, v in pred.items():
                nv = normalize_value(v)
                if k != concept and ((gt_norm == nv) or (gt_norm in nv) or (nv in gt_norm and len(nv) > 3)):
                    bound_to = k
                    break

        per_field[concept] = {
            "gt_value": gt_norm,
            "pred_value": pred_val,
            "field_match": field_match,
            "cba_match": cba_match,
            "misbinding": field_match and not cba_match,
            "bound_to": bound_to,
        }

    total = len(scored)
    if total == 0:
        return {"field_recall": 0, "cba_strict": 0, "cba_soft": 0, "delta": 0,
                "total": 0, "per_field": {}, "misbinding_count": 0}

    misbinding_count = sum(1 for d in per_field.values() if d["misbinding"])

    # CBA-soft
    cba_soft_score = 0
    for concept, detail in per_field.items():
        if detail["cba_match"]:
            cba_soft_score += 1.0
        elif detail["misbinding"] and detail["bound_to"]:
            gt_fam = CONCEPT_TO_FAMILY.get(concept, "")
            pred_fam = CONCEPT_TO_FAMILY.get(detail["bound_to"], "")
            cba_soft_score += 0.5 if gt_fam == pred_fam else 0.0

    return {
        "field_recall": field_correct / total,
        "cba_strict": cba_correct / total,
        "cba_soft": cba_soft_score / total,
        "delta": (field_correct - cba_correct) / total,
        "total": total,
        "scored_concepts": scored,
        "per_field": per_field,
        "misbinding_count": misbinding_count,
    }

print(f"Scoring functions ready ({len(EVAL_CONCEPTS)} concepts)")

Scoring functions ready (6 concepts)


In [ ]:
MAX_PAGES = 3  # Send first 3 pages to vision API
RENDER_DPI = 200

def pdf_to_images(pdf_path, max_pages=MAX_PAGES, dpi=RENDER_DPI):
    """Convert PDF pages to PIL images using PyMuPDF."""
    doc = fitz.open(pdf_path)
    images = []
    for page_num in range(min(len(doc), max_pages)):
        page = doc[page_num]
        mat = fitz.Matrix(dpi / 72, dpi / 72)
        pix = page.get_pixmap(matrix=mat)
        img = PILImage.frombytes("RGB", [pix.width, pix.height], pix.samples)
        images.append(img)
    doc.close()
    return images

def encode_image(img, max_width=1024):
    """Resize and base64-encode a PIL image."""
    if img.width > max_width:
        ratio = max_width / img.width
        img = img.resize((max_width, int(img.height * ratio)), PILImage.LANCZOS)
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.standard_b64encode(buf.getvalue()).decode("utf-8")

def extract_anthropic(pdf_path, model_id):
    """Extract via Anthropic vision API with multi-page support."""
    images = pdf_to_images(pdf_path)
    content = []
    for img in images:
        b64 = encode_image(img)
        content.append({"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": b64}})
    content.append({"type": "text", "text": "Extract all document-level fields from this FARA registration form."})

    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    message = client.messages.create(
        model=model_id,
        max_tokens=1024,
        system=FARA_SYSTEM_PROMPT, # Changed from ADBUY_SYSTEM_PROMPT
        messages=[{"role": "user", "content": content}],
    )
    raw = message.content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0]
    return json.loads(raw)

def extract_openai(pdf_path, model_id):
    """Extract via OpenAI vision API with multi-page support."""
    images = pdf_to_images(pdf_path)
    content = []
    for img in images:
        b64 = encode_image(img)
        content.append({"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}})
    content.append({"type": "text", "text": "Extract all document-level fields from this FARA registration form."})

    client = openai.OpenAI(api_key=OPENAI_API_KEY)
    response = client.chat.completions.create(
        model=model_id,
        max_tokens=1024,
        messages=[
            {"role": "system", "content": FARA_SYSTEM_PROMPT}, # Changed from ADBUY_SYSTEM_PROMPT
            {"role": "user", "content": content},
        ],
    )
    raw = response.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0]
    return json.loads(raw)

MODELS = {
    "haiku": {"provider": "anthropic", "model_id": "claude-haiku-4-5-20251001"},
    "gpt4o-mini": {"provider": "openai", "model_id": "gpt-4o-mini"},
}

def extract_fara(pdf_path, model_name):
    """Route to correct provider."""
    cfg = MODELS[model_name]
    if cfg["provider"] == "anthropic":
        return extract_anthropic(pdf_path, cfg["model_id"])
    else:
        return extract_openai(pdf_path, cfg["model_id"])

print(f"Extraction functions ready. Models: {list(MODELS.keys())}")
print(f"Samples available: {len(samples)}")

# Quick test render
if samples:
    test_imgs = pdf_to_images(samples[0]["pdf_path"])
    print(f"Test render: {len(test_imgs)} pages, first page {test_imgs[0].size}")
else:
    print("ERROR: No samples loaded. Check cell 2 output — likely PDF paths don't exist.")
    print(f"  all_docs loaded: {len(all_docs)}")

Extraction functions ready. Models: ['haiku', 'gpt4o-mini']
Samples available: 50
Test render: 3 pages, first page (1702, 2800)


In [ ]:
all_results = {}

for model_name in MODELS:
    print(f"\n{'='*60}")
    print(f"Running: {model_name} ({MODELS[model_name]['model_id']})")
    print(f"{'='*60}")

    results = []
    errors = []

    for i, s in enumerate(samples):
        predicted = None
        for attempt in range(5):
            try:
                predicted = extract_fara(s["pdf_path"], model_name)
                break
            except Exception as e:
                err_str = str(e)
                is_rate_limit = "429" in err_str or "rate_limit" in err_str.lower()
                if attempt < 4:
                    if is_rate_limit:
                        wait = min(10 * (2 ** attempt), 120)
                        print(f"  [{s['doc_id']}] Rate limited (attempt {attempt+1}). Waiting {wait}s...")
                    else:
                        wait = 2 ** (attempt + 1)
                        print(f"  [{s['doc_id']}] Attempt {attempt+1} failed: {e}. Retrying in {wait}s...")
                    time.sleep(wait)
                else:
                    print(f"  [{s['doc_id']}] FAILED after 5 attempts: {e}")
                    errors.append({"doc_id": s["doc_id"], "error": str(e)})

        if predicted is None:
            continue

        scores = score_document(s["ground_truth"], predicted)
        results.append({
            "doc_id": s["doc_id"],
            "filename": s["filename"],
            "predicted": predicted,
            "ground_truth": s["ground_truth"],
            "scores": scores,
        })

        mb = scores["misbinding_count"]
        status = "OK" if mb == 0 else f"MISBIND={mb}"
        if (i + 1) % 10 == 0 or i == 0 or mb > 0:
            print(f"  [{i+1:2d}/{len(samples)}] {s['doc_id']} — F1={scores['field_recall']:.3f}  CBA={scores['cba_strict']:.3f}  scored={scores['total']}  {status}")

        time.sleep(1.5 if MODELS[model_name]["provider"] == "anthropic" else 5.0)

    if results:
        avg_f1 = sum(r["scores"]["field_recall"] for r in results) / len(results)
        avg_cba = sum(r["scores"]["cba_strict"] for r in results) / len(results)
        avg_cba_soft = sum(r["scores"]["cba_soft"] for r in results) / len(results)
        avg_delta = sum(r["scores"]["delta"] for r in results) / len(results)
        total_misbindings = sum(r["scores"]["misbinding_count"] for r in results)
    else:
        avg_f1 = avg_cba = avg_cba_soft = avg_delta = total_misbindings = 0

    all_results[model_name] = {
        "results": results,
        "errors": errors,
        "avg_f1": round(avg_f1, 4),
        "avg_cba": round(avg_cba, 4),
        "avg_cba_soft": round(avg_cba_soft, 4),
        "avg_delta": round(avg_delta, 4),
        "total_misbindings": total_misbindings,
    }

    print(f"\n--- {model_name} Summary ---")
    print(f"  Field Recall:     {avg_f1:.4f}")
    print(f"  CBA-strict:   {avg_cba:.4f}")
    print(f"  CBA-soft:     {avg_cba_soft:.4f}")
    print(f"  Delta:        {avg_delta:.4f}")
    print(f"  Misbindings:  {total_misbindings}")
    print(f"  Errors:       {len(errors)}")

print(f"\n{'='*60}")
print("ALL EXPERIMENTS COMPLETE")
print(f"{'='*60}")


Running: haiku (claude-haiku-4-5-20251001)
  [ 1/50] fara_0000 — F1=0.667  CBA=0.667  scored=3  OK
  [10/50] fara_0009 — F1=1.000  CBA=1.000  scored=5  OK
  [20/50] fara_0019 — F1=1.000  CBA=1.000  scored=3  OK
  [28/50] fara_0027 — F1=0.667  CBA=0.333  scored=3  MISBIND=1
  [30/50] fara_0029 — F1=1.000  CBA=1.000  scored=4  OK
  [40/50] fara_0039 — F1=1.000  CBA=1.000  scored=5  OK
  [50/50] fara_0049 — F1=1.000  CBA=1.000  scored=4  OK

--- haiku Summary ---
  Field F1:     0.8357
  CBA-strict:   0.8290
  CBA-soft:     0.8323
  Delta:        0.0067
  Misbindings:  1
  Errors:       0

Running: gpt4o-mini (gpt-4o-mini)
  [ 1/50] fara_0000 — F1=0.667  CBA=0.667  scored=3  OK
  [fara_0008] Rate limited (attempt 1). Waiting 10s...
  [10/50] fara_0009 — F1=1.000  CBA=1.000  scored=5  OK
  [fara_0010] Rate limited (attempt 1). Waiting 10s...
  [fara_0011] Rate limited (attempt 1). Waiting 10s...
  [fara_0013] Rate limited (attempt 1). Waiting 10s...
  [fara_0016] Rate limited (attempt 1).

In [ ]:
print("=" * 75)
print("CROSS-MODEL COMPARISON — VRDU Registration Forms (FARA)")
print("=" * 75)

print(f"\n{'Model':<15} {'Field Recall':>10} {'CBA-strict':>12} {'CBA-soft':>10} {'Delta':>8} {'Misbindings':>13} {'Errors':>8}")
print("-" * 80)
for name, res in all_results.items():
    print(f"{name:<15} {res['avg_f1']:>10.4f} {res['avg_cba']:>12.4f} {res['avg_cba_soft']:>10.4f} {res['avg_delta']:>8.4f} {res['total_misbindings']:>13} {len(res['errors']):>8}")

total_misbindings = sum(r["total_misbindings"] for r in all_results.values())
print(f"\nTotal misbindings across all models: {total_misbindings}")

# Per-concept accuracy
print(f"\n{'='*75}")
print("PER-CONCEPT ACCURACY")
print(f"{'='*75}")

concept_stats = defaultdict(lambda: {"cba": 0, "field": 0, "total": 0, "misbindings": 0})
for model_name, res in all_results.items():
    for r in res["results"]:
        for concept, detail in r["scores"]["per_field"].items():
            cs = concept_stats[concept]
            cs["total"] += 1
            if detail["cba_match"]:
                cs["cba"] += 1
            if detail["field_match"]:
                cs["field"] += 1
            if detail["misbinding"]:
                cs["misbindings"] += 1

concept_rows = [(c, concept_stats[c]) for c in EVAL_CONCEPTS if concept_stats[c]["total"] > 0]
concept_rows.sort(key=lambda x: -x[1]["misbindings"])

print(f"\n{'Concept':<25} {'Family':<15} {'Field Acc':>10} {'CBA Acc':>10} {'Delta':>8} {'Misbindings':>13}")
print("-" * 85)
for concept, cs in concept_rows:
    f_acc = cs["field"] / cs["total"]
    c_acc = cs["cba"] / cs["total"]
    delta = f_acc - c_acc
    fam = CONCEPT_TO_FAMILY[concept]
    flag = " ***" if cs["misbindings"] > 3 else ""
    print(f"{concept:<25} {fam:<15} {f_acc:>10.3f} {c_acc:>10.3f} {delta:>8.3f} {cs['misbindings']:>13}{flag}")

CROSS-MODEL COMPARISON — VRDU Registration Forms (FARA)

Model             Field F1   CBA-strict   CBA-soft    Delta   Misbindings   Errors
--------------------------------------------------------------------------------
haiku               0.8357       0.8290     0.8323   0.0067             1        0
gpt4o-mini          0.7780       0.7730     0.7755   0.0050             1        0

Total misbindings across all models: 2

PER-CONCEPT ACCURACY

Concept                   Family           Field Acc    CBA Acc    Delta   Misbindings
-------------------------------------------------------------------------------------
registrant_name           names                0.920      0.900    0.020             2
registration_num          identifiers          0.867      0.867    0.000             0
foreign_principle_name    names                0.828      0.828    0.000             0
signer_name               names                0.855      0.855    0.000             0
signer_title              sig

This cell was identified as attempting to load an unrelated dataset (ad-buy forms) and redefine `EVAL_CONCEPTS`, which conflicts with the VRDU Registration Form (FARA) experiment. It has been removed to ensure the experiment's integrity.

In [ ]:
print("=" * 75)
print("MISBINDING CONFUSION PAIRS")
print("=" * 75)

confusion = Counter()
all_misbindings = []
for model_name, res in all_results.items():
    for r in res["results"]:
        for concept, detail in r["scores"]["per_field"].items():
            if detail["misbinding"] and detail["bound_to"]:
                confusion[(concept, detail["bound_to"])] += 1
                all_misbindings.append({
                    "model": model_name,
                    "doc_id": r["doc_id"],
                    "concept": concept,
                    "bound_to": detail["bound_to"],
                    "gt_value": detail["gt_value"],
                    "family_gt": CONCEPT_TO_FAMILY.get(concept, "?"),
                    "family_pred": CONCEPT_TO_FAMILY.get(detail["bound_to"], "?"),
                })

if confusion:
    print(f"\n{'Expected Concept':<25} {'Bound To':<25} {'Count':>6} {'Families':>30}")
    print("-" * 90)
    for (src, dst), count in confusion.most_common(20):
        fam_src = CONCEPT_TO_FAMILY.get(src, "?")
        fam_dst = CONCEPT_TO_FAMILY.get(dst, "?")
        same = "SAME-FAM" if fam_src == fam_dst else "CROSS-FAM"
        print(f"{src:<25} {dst:<25} {count:>6} {fam_src}->{fam_dst} ({same})")

    print(f"\n{'='*75}")
    print("FAMILY-LEVEL CONFUSION SUMMARY")
    print(f"{'='*75}")
    fam_confusion = Counter()
    for mb in all_misbindings:
        fam_confusion[(mb["family_gt"], mb["family_pred"])] += 1

    print(f"\n{'GT Family':<20} {'Pred Family':<20} {'Count':>6} {'Type':>12}")
    print("-" * 60)
    for (fgt, fpred), count in fam_confusion.most_common():
        same = "within-fam" if fgt == fpred else "cross-fam"
        print(f"{fgt:<20} {fpred:<20} {count:>6} {same:>12}")
else:
    print("No misbindings detected.")

MISBINDING CONFUSION PAIRS

Expected Concept          Bound To                   Count                       Families
------------------------------------------------------------------------------------------
registrant_name           foreign_principle_name         1 names->names (SAME-FAM)
registrant_name           signer_name                    1 names->names (SAME-FAM)

FAMILY-LEVEL CONFUSION SUMMARY

GT Family            Pred Family           Count         Type
------------------------------------------------------------
names                names                     2   within-fam


In [ ]:
export = {
    "experiment": "hindsight_vrdu_registration_cba",
    "domain": "government_registration",
    "dataset": "VRDU registration forms / FARA (google-research-datasets/vrdu)",
    "sample_size": len(samples),
    "total_concepts": len(EVAL_CONCEPTS),
    "ontology": FARA_ONTOLOGY,
    "models": {k: v for k, v in MODELS.items()},
    "scoring_method": "value-first CBA, GT-backed concepts only, containment matching for names",
    "results": {
        model_name: {
            "field_recall": res["avg_f1"],
            "cba_strict": res["avg_cba"],
            "cba_soft": res["avg_cba_soft"],
            "delta": res["avg_delta"],
            "total_misbindings": res["total_misbindings"],
            "num_errors": len(res["errors"]),
        }
        for model_name, res in all_results.items()
    },
    "confusion_pairs": [
        {"expected": src, "bound_to": dst, "count": count}
        for (src, dst), count in confusion.most_common(20)
    ] if confusion else [],
    "total_misbindings": total_misbindings,
}

output_path = "/tmp/hindsight_vrdu_registration_cba_results.json"
with open(output_path, "w") as f:
    json.dump(export, f, indent=2)
print(f"Results exported to: {output_path}")

# Cross-domain comparison
print(f"\n{'='*75}")
print("CROSS-DOMAIN COMPARISON (all experiments)")
print(f"{'='*75}")
print(f"\n{'Domain':<25} {'Dataset':<20} {'Misbindings':>12} {'Haiku Delta':>13} {'GPT4o-mini Delta':>17}")
print("-" * 90)
print(f"{'Paystubs':<25} {'Synthetic':<20} {'~5':>12} {'~0':>13} {'~0':>17}")
print(f"{'Receipts (SROIE)':<25} {'SROIE':<20} {'3':>12} {'0.000':>13} {'+0.015':>17}")
print(f"{'Receipts (CORD)':<25} {'CORD v2':<20} {'48':>12} {'-0.10':>13} {'~0':>17}")
print(f"{'W-2 Tax Forms':<25} {'Synthetic W-2':<20} {'421':>12} {'+0.128':>13} {'+0.094':>17}")
print(f"{'Employment Law':<25} {'LEDGAR':<20} {'350':>12} {'+0.207':>13} {'+0.260':>17}")
print(f"{'Commercial Law':<25} {'CUAD v1':<20} {'434':>12} {'+0.244':>13} {'+0.335':>17}")

haiku_d = all_results.get("haiku", {}).get("avg_delta", 0)
gpt_d = all_results.get("gpt4o-mini", {}).get("avg_delta", 0)
print(f"{'FARA Registration':<25} {'VRDU':<20} {total_misbindings:>12} {f'+{haiku_d:.3f}':>13} {f'+{gpt_d:.3f}':>17}")

Results exported to: /tmp/hindsight_vrdu_registration_cba_results.json

CROSS-DOMAIN COMPARISON (all experiments)

Domain                    Dataset               Misbindings   Haiku Delta  GPT4o-mini Delta
------------------------------------------------------------------------------------------
Paystubs                  Synthetic                      ~5            ~0                ~0
Receipts (SROIE)          SROIE                           3         0.000            +0.015
Receipts (CORD)           CORD v2                        48         -0.10                ~0
W-2 Tax Forms             Synthetic W-2                 421        +0.128            +0.094
Employment Law            LEDGAR                        350        +0.207            +0.260
Commercial Law            CUAD v1                       434        +0.244            +0.335
FARA Registration         VRDU                            2        +0.007            +0.005
